In [1]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
import pandas as pd
import numpy as np

from dataprocessing import data, data_noutliers

c:\Users\yzhen\OneDrive\Documents\MSPPM-DA\SP2025\ML\project - education\dataprocessing.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['school_state_region'] = data['school_state'].map(state_to_region)
c:\Users\yzhen\OneDrive\Documents\MSPPM-DA\SP2025\ML\project - education\dataprocessing.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['fully_funded'] = data['fully_funded'].map(binary_map)


In [2]:
data_noutliers

,projectid,school_state,school_metro,school_magnet,school_nlns,school_kipp,school_charter,school_charter_ready_promise,teacher_teach_for_america,teacher_ny_teaching_fellow,...,students_reached,eligible_double_your_impact_match,eligible_almost_home_match,date_posted,fully_funded,short_description,need_statement,essay,school_state_region,total_price_excluding_optional_support_scaled
0,77558a6eda151deee9a00553f7fccfc7,NY,urban,t,f,f,f,f,f,f,...,0.0,f,f,2002-09-13,1,I am a first year teacher assigned to the 6th ...,"The cost of this proposal is [price], includin...",I am a first year teacher assigned to the 6th ...,Northeast,-0.941535
1,82e536f14eadf2671a70e03416f695a3,NY,urban,t,f,f,f,f,f,f,...,0.0,f,f,2002-09-16,1,I just returned from 3 weeks in S. Africa. I w...,"The cost of this proposal is [price], includin...",I just returned from 3 weeks in S. Africa. I ...,Northeast,-1.436594
2,c7d251ab36c83155af3269afc8df0b63,NY,urban,f,f,f,f,f,f,f,...,0.0,f,f,2002-09-17,1,Hello! I am a fifth and sixth grade bilingual ...,"The cost of the books, ranging from ""El Castil...",Hello! I am a fifth and sixth grade bilingual ...,Northeast,-0.799919
3,fd5a8eccdbaa72f9d526dd6e97ca063d,NY,urban,f,f,f,f,f,t,f,...,0.0,f,f,2002-09-17,1,"Hi, my name is Sara Shenkan and I am a 4th gra...","The cost of this proposal is $476, including s...","Hi, my name is Sara Shenkan and I am a 4th gra...",Northeast,-0.175144
4,944d10e78d75a766de19bdd03aa41f87,NY,urban,f,f,f,f,f,f,f,...,0.0,f,f,2002-09-17,1,Greetings! I teach sixth grade history/social ...,"The cost of this proposal is [price], includin...",Greetings! I teach sixth grade history/social...,Northeast,-1.436594
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515103,a0e839f24645e3d6dcbd327f8441b043,NY,urban,f,f,f,f,f,f,f,...,60.0,f,f,2013-12-31,1,"On a typical day in my classroom, special need...",My students need an assortment of basic art su...,"On a typical day in my classroom, special need...",Northeast,0.668266
515104,f820ef3537f4445b0716244fae36f763,ID,urban,f,f,f,f,f,f,f,...,30.0,f,f,2013-12-31,0,Math and reading are two separate subjects in ...,My students need a large range of math stories...,Math and reading are two separate subjects in ...,West,-0.711284
515105,95ee208a51831edffa7cc2e0aa3e83cd,AZ,urban,f,f,f,f,f,f,f,...,98.0,f,f,2013-12-31,1,Remember the first book you ever loved? How yo...,My students need books to improve their readin...,Remember the first book you ever loved? How yo...,West,-1.068441
515106,383500f017fea60562ba737b051e1d21,MN,suburban,f,f,f,f,f,f,f,...,29.0,f,f,2013-12-31,1,"I will be honest with you, I'm horrible at pre...",My students need funding for a workshop at The...,"I will be honest with you, I'm horrible at pre...",Midwest,0.526983


In [3]:
data

,projectid,school_state,school_metro,school_magnet,school_nlns,school_kipp,school_charter,school_charter_ready_promise,teacher_teach_for_america,teacher_ny_teaching_fellow,...,total_price_including_optional_support,students_reached,eligible_double_your_impact_match,eligible_almost_home_match,date_posted,fully_funded,short_description,need_statement,essay,school_state_region
0,77558a6eda151deee9a00553f7fccfc7,NY,urban,t,f,f,f,f,f,f,...,279.27,0.0,f,f,2002-09-13,1,I am a first year teacher assigned to the 6th ...,"The cost of this proposal is [price], includin...",I am a first year teacher assigned to the 6th ...,Northeast
1,82e536f14eadf2671a70e03416f695a3,NY,urban,t,f,f,f,f,f,f,...,152.44,0.0,f,f,2002-09-16,1,I just returned from 3 weeks in S. Africa. I w...,"The cost of this proposal is [price], includin...",I just returned from 3 weeks in S. Africa. I ...,Northeast
2,e02da37beb332eb66c2d2ba989c597ad,NY,urban,f,f,f,f,f,f,f,...,1376.83,0.0,f,f,2002-09-17,1,I teach economics to 25 students at Satellite ...,"The cost of this proposal is $1377, including ...",I teach economics to 25 students at Satellite ...,Northeast
3,c7d251ab36c83155af3269afc8df0b63,NY,urban,f,f,f,f,f,f,f,...,315.55,0.0,f,f,2002-09-17,1,Hello! I am a fifth and sixth grade bilingual ...,"The cost of the books, ranging from ""El Castil...",Hello! I am a fifth and sixth grade bilingual ...,Northeast
4,fd5a8eccdbaa72f9d526dd6e97ca063d,NY,urban,f,f,f,f,f,t,f,...,475.61,0.0,f,f,2002-09-17,1,"Hi, my name is Sara Shenkan and I am a 4th gra...","The cost of this proposal is $476, including s...","Hi, my name is Sara Shenkan and I am a 4th gra...",Northeast
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
542219,f820ef3537f4445b0716244fae36f763,ID,urban,f,f,f,f,f,f,f,...,326.32,30.0,f,f,2013-12-31,0,Math and reading are two separate subjects in ...,My students need a large range of math stories...,Math and reading are two separate subjects in ...,West
542220,95ee208a51831edffa7cc2e0aa3e83cd,AZ,urban,f,f,f,f,f,f,f,...,238.05,98.0,f,f,2013-12-31,1,Remember the first book you ever loved? How yo...,My students need books to improve their readin...,Remember the first book you ever loved? How yo...,West
542221,383500f017fea60562ba737b051e1d21,MN,suburban,f,f,f,f,f,f,f,...,632.35,29.0,f,f,2013-12-31,1,"I will be honest with you, I'm horrible at pre...",My students need funding for a workshop at The...,"I will be honest with you, I'm horrible at pre...",Midwest
542222,efa879a124262ea850736f9f9a54ff7b,CA,rural,f,f,f,f,f,f,f,...,1222.45,22.0,f,f,2013-12-31,0,"My students are curious, not only about lesson...",My students need expenses for admission and tr...,"My students are curious, not only about lesson...",West


**School Metro & Poverty Levels**

Baseline

In [4]:
features = ['school_metro','poverty_level']

In [6]:
#one hot encoding

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data_noutliers[features])

#training

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data_noutliers['fully_funded'], test_size=0.2, random_state=42,stratify=data_noutliers['fully_funded'])

clf = HistGradientBoostingClassifier().fit(X_train,y_train)

#testing

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     29456
           1       0.71      1.00      0.83     73566

    accuracy                           0.71    103022
   macro avg       0.36      0.50      0.42    103022
weighted avg       0.51      0.71      0.59    103022

Specificity: 0.00
ROC-AUC: 0.5



c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Random Under Sampler for Class Unbalances

In [7]:
#training

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

clf_rus = HistGradientBoostingClassifier().fit(X_train_rus,y_train_rus)
y_pred_rus = clf_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.34      0.53      0.41     29456
           1       0.75      0.58      0.66     73566

    accuracy                           0.57    103022
   macro avg       0.54      0.55      0.53    103022
weighted avg       0.63      0.57      0.59    103022

Specificity: 0.53
ROC-AUC: 0.5



In [8]:
#checking distribution

print(f"\nBefore Undersampling - Training examples: {len(X_train)}")
print(f"Class distribution: {np.bincount(y_train)}")

print(f"\nAfter Undersampling - Training examples: {len(X_train_rus)}")
print(f"Class distribution: {np.bincount(y_train_rus)}")


Before Undersampling - Training examples: 412086
Class distribution: [117823 294263]

After Undersampling - Training examples: 235646
Class distribution: [117823 117823]


**RUS: Adding Resource Type & Primary Subject**

In [9]:
features = ['school_metro','poverty_level','resource_type','primary_focus_subject']

In [11]:
#one hot encoding

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data_noutliers[features])

#training

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data_noutliers['fully_funded'], test_size=0.2, random_state=42,stratify=data_noutliers['fully_funded'])

#undersampling

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

clf_rus = HistGradientBoostingClassifier().fit(X_train_rus,y_train_rus)
y_pred_rus = clf_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.36      0.61      0.45     29456
           1       0.78      0.56      0.65     73566

    accuracy                           0.57    103022
   macro avg       0.57      0.58      0.55    103022
weighted avg       0.66      0.57      0.59    103022

Specificity: 0.61
ROC-AUC: 0.5



**RUS: Add Students Reached + Funding Request Amt**

In [12]:
features = ['school_metro','poverty_level','resource_type','primary_focus_subject']

In [15]:
#one hot encoding and adding numerical features

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data_noutliers[features])

#add numerical features
features_encoded = features_encoded.tolist()

for i in range(len(features_encoded)):
    features_encoded[i] = features_encoded[i] + [data_noutliers.loc[i,'students_reached']] + [data_noutliers.loc[i,'total_price_excluding_optional_support']]

features_encoded = np.array(features_encoded)

In [16]:
#training 

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data_noutliers['fully_funded'], test_size=0.2, random_state=42,stratify=data_noutliers['fully_funded'])

#undersampling

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

clf_rus = HistGradientBoostingClassifier().fit(X_train_rus,y_train_rus)
y_pred_rus = clf_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.40      0.72      0.51     29456
           1       0.83      0.56      0.67     73566

    accuracy                           0.61    103022
   macro avg       0.62      0.64      0.59    103022
weighted avg       0.71      0.61      0.63    103022

Specificity: 0.72
ROC-AUC: 0.5



**RUS: Funding Request Amt + Student Reached + Matching Status + Resource Type**

In [17]:
features = ['eligible_double_your_impact_match','eligible_almost_home_match','primary_focus_subject','resource_type']

In [ ]:
#one hot encoding

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data_noutliers[features])

#add numerical features
features_encoded = features_encoded.tolist()

for i in range(len(features_encoded)):
    features_encoded[i] = features_encoded[i] + [data_noutliers.loc[i,'students_reached']] + [data_noutliers.loc[i,'total_price_excluding_optional_support']]

features_encoded = np.array(features_encoded)

#training 

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data_noutliers['fully_funded'], test_size=0.2, random_state=42,stratify=data_noutliers['fully_funded'])

#undersampling

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

clf_rus = HistGradientBoostingClassifier().fit(X_train_rus,y_train_rus)
y_pred_rus = clf_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.40      0.70      0.51     29456
           1       0.83      0.58      0.68     73566

    accuracy                           0.62    103022
   macro avg       0.61      0.64      0.60    103022
weighted avg       0.71      0.62      0.63    103022

Specificity: 0.70
ROC-AUC: 0.5



**RUS: School Type**

In [58]:
features = ['school_magnet','school_nlns','school_kipp','school_charter','school_charter_ready_promise','teacher_teach_for_america','teacher_ny_teaching_fellow']

In [19]:
ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data_noutliers[features])

#training 

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data_noutliers['fully_funded'], test_size=0.2, random_state=42,stratify=data_noutliers['fully_funded'])

#undersampling

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

clf_rus = HistGradientBoostingClassifier().fit(X_train_rus,y_train_rus)
y_pred_rus = clf_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.36      0.55      0.44     29456
           1       0.77      0.62      0.68     73566

    accuracy                           0.60    103022
   macro avg       0.57      0.58      0.56    103022
weighted avg       0.65      0.60      0.61    103022

Specificity: 0.55
ROC-AUC: 0.5

